# NOTEBOOK DE COLAB, AQUI NO ESTA EL DATASET DESCARGADO

In [ ]:
import os, pickle
d = "/content/drive/MyDrive/radar-trafico/data/raw/session2_center"

# 1. ¿El vídeo está completo?
print(os.path.getsize(f"{d}/video.avi") == 9508744968)

# 2. ¿El pkl se abre?
with open(f"{d}/gt_data.pkl", "rb") as f:
    gt = pickle.load(f, encoding="latin1")
print(type(gt), gt.keys() if isinstance(gt, dict) else len(gt))

In [ ]:
!mkdir -p /content/drive/MyDrive/radar-trafico/data/raw
!cp -r /content/brno/2016-ITS-BrnoCompSpeed/dataset/session2_center /content/drive/MyDrive/radar-trafico/data/raw/

In [ ]:
print(gt["fps"])
car = gt["cars"][39]
print(car.keys())
print(car)

In [ ]:
import numpy as np
import cv2, matplotlib.pyplot as plt

screen = cv2.cvtColor(cv2.imread(f"{d}/screen.png"), cv2.COLOR_BGR2RGB)
mask = cv2.imread(f"{d}/video_mask.png", cv2.IMREAD_GRAYSCALE)
print(screen.shape, mask.shape, np.unique(mask))

fig, ax = plt.subplots(1, 2, figsize=(16, 5))
ax[0].imshow(screen); ax[0].set_title("screen.png")
ax[1].imshow(screen); ax[1].imshow(mask, alpha=0.4, cmap="Reds"); ax[1].set_title("máscara superpuesta")
plt.show()

In [ ]:
car = gt["cars"][600]
print(car.keys())
print(car["acceleration"])
print(car["intersections"])
print(car["carId"])
print(car["timeIntersectionLastShifted"])
print(car["valid"])
print(car["laneIndex"])
print(car["speed"])

In [ ]:
intersections = car["intersections"]
print(intersections)

def tiempoCoche(intersections):         # tiempo que tarda en cruzar las lineas 0 y 4 del dataset

    tiempos = []
    for intersection in intersections:
        tiempos.append(intersection["videoTime"])

    linea0 = tiempos[1]
    linea4 = tiempos[0]

    return linea4, linea0

print(tiempoCoche(intersections))

In [ ]:
from posix import write
FRAMES_POR_SEGUNDO = 100

def leer_frames(rutaVideo, tInicio, tFinal):
    cap = cv2.VideoCapture(rutaVideo)

    frameInicio = round(tInicio * FRAMES_POR_SEGUNDO)
    frameFinal = round(tFinal * FRAMES_POR_SEGUNDO)

    cap.set(cv2.CAP_PROP_POS_FRAMES, frameInicio)

    for i in range(frameInicio, frameFinal):
        ok, frame = cap.read()

        if not ok:
          break

        yield i, i / FRAMES_POR_SEGUNDO, frame

    cap.release()

def mostrar_clip(rutaVideo, tInicio, tFinal, modelo, salida="clip.mp4"):
  writer = None

  for i, t, frame in leer_frames(rutaVideo, tInicio, tFinal):

    resultado = modelo(frame, classes=[2, 3, 5, 7], verbose=False)[0]  # importante, filtro para vehiculos
    frame = resultado.plot()

    if writer is None:
      h, w = frame.shape[:2]
      writer = cv2.VideoWriter(salida, cv2.VideoWriter_fourcc(*"mp4v"), 100, (w, h))
    writer.write(frame)
  writer.release()

car = gt["cars"][600]
intersections = car["intersections"]
tInicio, tFinal = tiempoCoche(intersections)

In [ ]:
!pip install ultralytics

from IPython.display import Video
from ultralytics import YOLO
!rm -f clip.mp4 clip_h264.mp4

modelo = YOLO("yolo11n.pt")

mostrar_clip("/content/drive/MyDrive/radar-trafico/data/raw/session2_center/video.avi", tInicio, tFinal, modelo)

!ffprobe -v error -count_frames -select_streams v:0 -show_entries stream=nb_read_frames -of csv=p=0 clip.mp4
!ffmpeg -y -loglevel error -i clip.mp4 -c:v libx264 clip_h264.mp4
Video("clip_h264.mp4", embed=True)

In [ ]:
from IPython.display import Video, display

modelo = YOLO("yolo11n.pt")

import random
random.seed(42)

candidatos = [
    c for c in gt["cars"]
    if c["valid"]
    and len(c["intersections"]) == 2
    and max(i["videoTime"] for i in c["intersections"]) < 1200   # solo el bloque de ajuste
]

muestra = random.sample(candidatos, 10)

NO_MOSTRAR_VIDEOS = False  # no imprime los video y me deja guardar el notebook

if NO_MOSTRAR_VIDEOS:
  for car in muestra:
      intersections = car["intersections"]
      if len(intersections) == 2:
          cid = car["carId"]
          tInicio, tFinal = tiempoCoche(intersections)
          mostrar_clip("/content/drive/MyDrive/radar-trafico/data/raw/session2_center/video.avi", tInicio, tFinal, modelo, salida=f"clip_{cid}.mp4")
          !ffmpeg -y -loglevel error -i clip_{cid}.mp4 -c:v libx264 clip_{cid}_h264.mp4
          display(Video(f"clip_{cid}_h264.mp4", embed=True))

In [ ]:
display(Video(f"clip_65_h264.mp4", embed=True))

In [ ]:
CLIPS_TRACKING = {
    "simple" : {
        "carId" : 72,
        "margen" : 0.5,
        "nota" : "turismo solo, carril izquierdo, sin mas coches"
    },

    "paralelo" : {
        "carId" : 350,
        "margen" : 0.5,
        "nota" : "varios coches en escena, en ambos carriles"
    },

    "adelantamiento" : {
        "carId" : 65,
        "margen" : 0.5,
        "nota" : "varios coches en ambos carriles, coche adelantando camion por izquierda"
    }
}

In [ ]:
def coche_por_id(gt, carId):
    for c in gt["cars"]:
        if c["carId"] == carId:
            return c
    raise ValueError(f"No existe el carId {carId}")

In [ ]:
car = coche_por_id(gt, CLIPS_TRACKING["paralelo"]["carId"])
tInicio, tFinal = tiempoCoche(car["intersections"])

ruta = "/content/drive/MyDrive/radar-trafico/data/raw/session2_center/video.avi"

gen = leer_frames(ruta, tInicio, tFinal)

_, tA, frameA = next(gen)
_, tB, frameB = next(gen)

resA = modelo(frameA, classes=[2, 3, 5, 7], verbose=False)[0]
resB = modelo(frameB, classes=[2, 3, 5, 7], verbose=False)[0]

print(resA.boxes)


In [ ]:
import pandas as pd

ruta = "/content/drive/MyDrive/radar-trafico/data/raw/session2_center/video.avi"

mask = cv2.imread(f"{d}/video_mask.png", cv2.IMREAD_GRAYSCALE)
car = coche_por_id(gt, CLIPS_TRACKING["paralelo"]["carId"])
tInicio, tFinal = tiempoCoche(car["intersections"])

def detectar_clip(ruta, tInicio, tFinal, modelo, mask, clases=(2, 3, 5, 7)):
    filas = []
    for idx, t, frame in leer_frames(ruta, tInicio, tFinal):
        res = modelo.track(frame, persist=True, tracker="bytetrack.yaml",
                   classes=list(clases), verbose=False)[0]
        cajas = res.boxes.xyxy.cpu().numpy()
        confianza = res.boxes.conf.cpu(). numpy()
        clase = res.boxes.cls.cpu().numpy().astype(int)

        if res.boxes.id is None:
            ids = np.full(len(cajas), -1)
        else:
            ids = res.boxes.id.cpu().numpy().astype(int)

        for (x1, y1, x2, y2), conf, cls, tids in zip(cajas, confianza, clase, ids):
            px = min(int((x1 + x2) / 2), mask.shape[1] - 1)
            py = min(int(y2), mask.shape[0] - 1)

            if mask[py, px] > 0:
                filas.append({
                    "frame": idx, "t": t,
                    "x1": x1, "y1": y1, "x2": x2, "y2": y2,
                    "conf": conf, "cls": cls,
                    "id" : tids,
                })

    return pd.DataFrame(filas)

df = detectar_clip(ruta, tInicio, tFinal, modelo, mask)
print(len(df), "cajas en", df["frame"].nunique(), "frames")
print(df.groupby("frame").size().value_counts())

In [ ]:
COLORES = {2: (0, 255, 0), 3: (255, 0, 255), 5: (0, 165, 255), 7: (0, 0, 255)}  # BGR: coche verde, camión rojo

def dibujar_df(ruta, tInicio, tFinal, df, salida="detecciones.mp4"):
    writer = None
    for idx, t, frame in leer_frames(ruta, tInicio, tFinal):

        for _, f in df[df["frame"] == idx].iterrows():
            x1, y1, x2, y2 = int(f.x1), int(f.y1), int(f.x2), int(f.y2)
            color = COLORES.get(int(f.cls), (255, 255, 255))
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.circle(frame, ((x1 + x2) // 2, y2), 6, color, -1)
            cv2.putText(frame, f"id {int(f.id)} {int(f.cls)} {f.conf:.2f} ", (x1, y1 - 8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

        n = (df["frame"] == idx).sum()
        cv2.putText(frame, f"t={t:.2f}s  cajas={n}", (30, 60),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 255), 3)

        if writer is None:
            h, w = frame.shape[:2]
            writer = cv2.VideoWriter(salida, cv2.VideoWriter_fourcc(*"mp4v"), 25, (w, h))
            assert writer.isOpened()
        writer.write(frame)
    writer.release()

In [ ]:
dibujar_df(ruta, tInicio, tFinal, df)

In [ ]:
!ffmpeg -y -loglevel error -i detecciones.mp4 -c:v libx264 detecciones_h264.mp4
display(Video("detecciones_h264.mp4", embed=True))

In [ ]:
df = detectar_clip(ruta, tInicio, tFinal, modelo, mask)

def cajas_de_frame(df, frame):
    return df[df["frame"] == frame][["x1", "y1", "x2", "y2"]].to_numpy()

def iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b

    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)

    ancho = max(0, ix2 - ix1)
    alto = max(0, iy2 - iy1)

    inter = ancho * alto
    area_a = (ax2 - ax1) * (ay2 - ay1)
    area_b = (bx2 - bx1) * (by2 - by1)

    return inter / (area_a + area_b - inter)

In [ ]:
import numpy as np

modelo = YOLO("yolo11n.pt")

car = coche_por_id(gt, CLIPS_TRACKING["paralelo"]["carId"])
tInicio, tFinal = tiempoCoche(car["intersections"])
df = detectar_clip(ruta, tInicio, tFinal, modelo, mask)

dibujar_df(ruta, tInicio, tFinal, df)

!ffmpeg -y -loglevel error -i detecciones.mp4 -c:v libx264 detecciones_h264.mp4
display(Video("detecciones_h264.mp4", embed=True))

In [ ]:
import ultralytics, os
ruta_cfg = os.path.join(os.path.dirname(ultralytics.__file__), "cfg", "trackers", "bytetrack.yaml")
print(open(ruta_cfg).read())

In [ ]:
print(df["id"].value_counts())
resumen = df[df["id"] >= 0].groupby("id")["frame"].agg(["min", "max", "count"])
resumen["huecos"] = (resumen["max"] - resumen["min"] + 1) - resumen["count"]
print(resumen)